In [ ]:
from __future__ import (absolute_import, division,
                        print_function, unicode_literals)

import warnings
warnings.simplefilter('ignore')

# general purpose packages
import pandas as pd
import numpy as np
import os
import json
import time
import re
import csv
import subprocess
import sys

import scipy.stats as stats
import statsmodels.stats as smstats
from statsmodels.stats.multitest import multipletests

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
 
from dotenv import load_dotenv
from pathlib import Path

from multiprocessing import Process, Manager, Pool
import multiprocessing
from functools import partial

from collections import Counter

import seaborn as sns; sns.set()

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
matplotlib.rcParams['backend'] = "Qt5Agg"
import matplotlib.ticker as ticker
from matplotlib.ticker import FuncFormatter

from IPython.display import display, Image

from adjustText import adjust_text
import builtins
%matplotlib inline

# for normalization
from sklearn.linear_model import QuantileRegressor

# for survival analysis
import sklearn
from sklearn import set_config

from statsmodels.regression.quantile_regression import QuantReg

# for working with yaml files
import ruamel.yaml

import itertools

# for working with .toml
import tomli_w

In [ ]:
def get_pvalue_star(pval, thr=0.05):
    if thr == 0.05:
        if pval < 0.001:
            return "***"
        elif pval < 0.01:
            return "**"
        elif pval < 0.05:
            return "*"
        else:
            return ""
    elif thr == 0.1:
        if pval < 0.001:
            return "***"
        elif pval < 0.01:
            return "**"
        elif pval < 0.1:
            return "*"
        else:
            return ""

In [ ]:
# 1. Load the environment variables
load_dotenv("APA_localization.scicore.env")

# 2. Reconstruct the subdirs dictionary
subdirs = {
    "lab_group_dir": os.getenv("LAB_GROUP_DIR"),
    "raw_sequencing_data_dir": os.getenv("RAW_SEQUENCING_DATA_DIR"),
    "main_project_dir": os.getenv("MAIN_PROJECT_DIR"),
    "wf_dir": os.getenv("WF_DIR"),
    "UCSCtracks_dir": os.getenv("UCSC_TRACKS_DIR"),
    "UCSCtracks_trackfiles_dir": os.getenv("UCSC_TRACKFILES_DIR"),
    "UCSCtracks_trackhubs_dir": os.getenv("UCSC_TRACKHUBS_DIR"),
    "human_annotation_dir": os.getenv("HUMAN_ANNOTATION_DIR"),
    "mouse_annotation_dir": os.getenv("MOUSE_ANNOTATION_DIR"),
    "shared_project_dir": os.getenv("SHARED_PROJECT_DIR"),
    "temp_dir": os.getenv("TEMP_DIR"),
    "slurm_dir": os.getenv("SLURM_DIR"),
    "slurm_scripts_dir": os.getenv("SLURM_SCRIPTS_DIR"),
    "figures_dir": os.getenv("FIGURES_DIR"),
    "tables_dir": os.getenv("TABLES_DIR"),
    "fastq_dir": os.getenv("FASTQ_DIR"),
    "metadata_dir": os.getenv("METADATA_DIR"),
    "external_data_dir": os.getenv("EXTERNAL_DATA_DIR"),
    "wf_runs_dir": os.getenv("WF_RUNS_DIR"),
    "pod5_dir": os.getenv("POD5_DIR"), # nanopore-specific
    "dorado_models_dir": os.getenv("DORADO_MODELS_DIR"), # nanopore-specific
    "nanoflowz_dir": os.getenv("NANOFLOWZ_DIR"), # nanopore-specific
}

# 3. Reconstruct the file_paths dictionary
file_paths = {
    "human_genome_file": os.getenv("HUMAN_GENOME_FILE"),
    "human_chrom_sizes_file": os.getenv("HUMAN_CHROM_SIZES_FILE"),
    "human_annotation_file": os.getenv("HUMAN_ANNOTATION_FILE"),
    "human_basic_annotation_file": os.getenv("HUMAN_BASIC_ANNOTATION_FILE"),
    "human_polyAsite_atlas": os.getenv("HUMAN_POLYASITE_ATLAS"),
    "human_tandem_PAS": os.getenv("HUMAN_TANDEM_PAS"),
    "human_exonic_segments_gtf": os.getenv("HUMAN_EXONIC_SEGMENTS_GTF"),
    "human_exonic_segments_bed": os.getenv("HUMAN_EXONIC_SEGMENTS_BED"),
    "dorado_executor": os.getenv("DORADO_EXECUTOR"), # nanopore-specific
    "dorado_executor_v2_0": os.getenv("DORADO_EXECUTOR_v2_0"), # nanopore-specific
}

# 4. Safely create all subdirectories
# Using os.makedirs is highly preferred over os.system('mkdir -p')
# because it avoids opening a subshell and handles permissions gracefully in pure Python.
for path in subdirs.values():
    if path:  # Safety check to ensure the variable was actually found in the .env
        os.makedirs(path, exist_ok=True)

print("Environment loaded and directories verified.")

# Make custom genome file with ERCC spike-ins (Lexogene)

In [ ]:
# WF_version = 'v1p4_ONT_FACSorganelles_Aleksei_v2_full'
WF_version = 'v2p0_ONT_FACSorganelles_Aleksei_v1'

dir_path = subdirs['wf_runs_dir']+WF_version+'/'
(Path(dir_path)).mkdir(parents=True, exist_ok=True) # create subdirectory for this version

In [ ]:
ref_genome_dir_path = os.path.join(dir_path, 'reference_genome')
(Path(ref_genome_dir_path)).mkdir(parents=True, exist_ok=True) # create subdirectory

In [ ]:
spike_ins_fasta = os.path.join(subdirs['external_data_dir'],'spike_ins','SIRV_Set3_Norm_Sequences_20250402','SIRV_ERCC_multi-fasta_20250402.fasta')
spike_ins_gtf = os.path.join(subdirs['external_data_dir'],'spike_ins','SIRV_Set3_Norm_Sequences_20250402','SIRV_ERCC_multi-fasta-annotation_20250402.gtf')

In [ ]:
# execute the following in command line to create a custom reference genome file
command = 'cat '+file_paths['human_genome_file']+' '+spike_ins_fasta+' > '+os.path.join(ref_genome_dir_path, 'genome_with_spikeINs.fasta')
print(command)

In [ ]:
gtf = pd.read_csv(file_paths['human_basic_annotation_file'],delimiter="\t",index_col=None,header=None,comment='#')

In [ ]:
construct_gtf = pd.read_csv(spike_ins_gtf,delimiter="\t",index_col=None,header=None,comment='#')

In [ ]:
gtf_df_compiled = pd.concat([gtf,construct_gtf]).reset_index(drop=True) # appended constructs

In [ ]:
gtf_df_compiled = gtf_df_compiled.sort_values([0,3]).reset_index(drop=True) # make it directly IGV-compatible

In [ ]:
gtf_df_compiled.to_csv(os.path.join(ref_genome_dir_path, 'compiled_genome_annotation.gtf'), sep=str('\t'),header=False,index=None,quoting=csv.QUOTE_NONE)

# Prepare start samples for nanoflowz

In [ ]:
# we extract the paths to .pod5 files

os.system("""find """+subdirs['pod5_dir']+""" -name '*.pod5' > """+subdirs['temp_dir']+"""pod5_files.tsv""")

In [ ]:
pod5_file_paths = pd.read_csv(subdirs['temp_dir']+'pod5_files.tsv',delimiter="\t",
                                   index_col=None,header=None)
pod5_file_paths.columns = ['pod5']
pod5_file_paths['sample_id'] = pod5_file_paths.apply(lambda x:x['pod5'].split('/')[-2],1)

# we are only interested in barcodes 01-12
sel_sample_ids = range(1,13)
sel_sample_ids = [('barcode0'+str(elem) if len(str(elem))==1 else 'barcode'+str(elem)) for elem in sel_sample_ids]
pod5_file_paths = pod5_file_paths.loc[pod5_file_paths['sample_id'].isin(sel_sample_ids)].reset_index(drop=True)

In [ ]:
# WF_version = 'v1p4_ONT_FACSorganelles_Aleksei_v2_full' # v1p4 at the beginning indicated Dorado version, while v2/v3 at the end indicates the particular version of the analysis
WF_version = 'v2p0_ONT_FACSorganelles_Aleksei_v1'

dir_path = subdirs['wf_runs_dir']+WF_version+'/'
(Path(dir_path)).mkdir(parents=True, exist_ok=True) # create subdirectory

# most important for A-seq3 - PAQR project, optional
# selected_samples = ['barcode0'+str(elem) for elem in [1,2,3,4]]
# pod5_file_paths = pod5_file_paths.loc[pod5_file_paths['sample_id'].isin(selected_samples)].reset_index(drop=True) 

pod5_file_paths[['sample_id','pod5']].to_csv(dir_path+'start_samples.tsv', sep=str('\t'),header=True,index=None,quoting=csv.QUOTE_NONE)

# Configuring nanoflowz execution

We've created a conda env "nextflow" to execute nanoflowz:

```bash
conda create --name nextflow bioconda::nextflow
conda activate nextflow
```

## v1p4_ONT_FACSorganelles_Aleksei_v2_full

In [ ]:
WF_version = 'v1p4_ONT_FACSorganelles_Aleksei_v2_full'
dir_path = subdirs['wf_runs_dir']+WF_version+'/'

run_dir = subdirs['wf_runs_dir']+WF_version+'/'

# create a .toml file for polyA tail profiling
polyA_toml_output_path = os.path.join(dir_path,'polyA_config.toml')
(Path(polyA_toml_output_path).parent).mkdir(parents=True, exist_ok=True) # create subdir, just in case

config_data = {
    "tail": {
        "tail_interrupt_length": 1
    }
}
with open(polyA_toml_output_path, "wb") as f:
    tomli_w.dump(config_data, f)

# define pipeline configuration
json_output_path = dir_path+'run_params.json'
json_file_dir = Path(json_output_path).parent
json_file_dir.mkdir(parents=True, exist_ok=True)

# define custom parameters for nanoflowz run

json_params_content = {
    "tsv": str(Path(dir_path+'start_samples.tsv').resolve()),
    "rundir": str(Path(run_dir).resolve()),
    "dorado": str(file_paths['dorado_executor']),
    "model": str(os.path.join(subdirs['dorado_models_dir'], 'dna_r10.4.1_e8.2_400bps_sup@v5.2.0')),
    "polyA": str(polyA_toml_output_path),
    "ref": str(file_paths['human_genome_file']),
    "reference_gtf": str(file_paths['human_basic_annotation_file']),
    "shift_ambiguous_cs": False, # <- here we don't enable the shifting of cleavage site positions!
    "check_gtf_compatibility": False, # no need, there are too many scaffolds in the genome that are not present in the used basic annotation
}
    
params_file = Path(json_output_path)
    
with open(params_file, "w") as f:
    json.dump(json_params_content, f, indent=4)

# run the printed command from any directory under the LOGIN node
# note that it also includes '-resume' argument to make sure that by default it won't be executed from the beginning
cmd = 'nextflow run '+\
subdirs['nanoflowz_dir']+'main.nf'+\
' -params-file '+str(params_file.resolve())+\
' -profile conda'+\
' -resume'
print(cmd)

In [ ]:
# now, try also with shifted cleavage site positions

# define pipeline configuration
json_output_path = dir_path+'run_params.ShiftedCleavageSites.json'
json_file_dir = Path(json_output_path).parent
json_file_dir.mkdir(parents=True, exist_ok=True)

# define custom parameters for nanoflowz run

json_params_content = {
    "tsv": str(Path(dir_path+'start_samples.tsv').resolve()),
    "rundir": str(Path(run_dir).resolve()),
    "dorado": str(file_paths['dorado_executor']),
    "model": str(os.path.join(subdirs['dorado_models_dir'], 'dna_r10.4.1_e8.2_400bps_sup@v5.2.0')),
    "polyA": str(polyA_toml_output_path),
    "ref": str(file_paths['human_genome_file']),
    "reference_gtf": str(file_paths['human_basic_annotation_file']),
    "shift_ambiguous_cs": True, # <- here we enable the shifting!
    "outdir": os.path.join(run_dir,'results_ShiftedCleavageSites'), # <- here
    "check_gtf_compatibility": False, # no need, there are too many scaffolds in the genome that are not present in the used basic annotation
}
    
params_file = Path(json_output_path)
    
with open(params_file, "w") as f:
    json.dump(json_params_content, f, indent=4)

# run the printed command from any directory under the LOGIN node
# note that it also includes '-resume' argument to make sure that by default it won't be executed from the beginning
cmd = 'nextflow run '+\
subdirs['nanoflowz_dir']+'main.nf'+\
' -params-file '+str(params_file.resolve())+\
' -profile conda'+\
' -resume'
print(cmd)

In [ ]:
# define pipeline configuration
json_output_path = dir_path+'run_params.alignment_with_SpikeIns.json'
json_file_dir = Path(json_output_path).parent
json_file_dir.mkdir(parents=True, exist_ok=True)

# define custom parameters for nanoflowz run

json_params_content = {
    "tsv": str(Path(dir_path+'start_samples.tsv').resolve()),
    "rundir": str(Path(run_dir).resolve()),
    "dorado": str(file_paths['dorado_executor']),
    "model": str(os.path.join(subdirs['dorado_models_dir'], 'dna_r10.4.1_e8.2_400bps_sup@v5.2.0')),
    "polyA": str(polyA_toml_output_path),
    "ref": str(os.path.join(run_dir,'reference_genome', 'genome_with_spikeINs.fasta')), # customly made genome which includes spike-ins
    "reference_gtf": str(os.path.join(run_dir,'reference_genome', 'compiled_genome_annotation.gtf')), # customly made genome which includes spike-ins
    "shift_ambiguous_cs": False, # <- here we don't enable the shifting of cleavage site positions!
    "outdir": os.path.join(run_dir,'results_alignment_with_SpikeIns'), # <- here !
    "check_gtf_compatibility": False, # no need, there are too many scaffolds in the genome that are not present in the used basic annotation
    "viz_gene_list": "ERCC-00130 ERCC-00096 ERCC-00002", # visualize raw signal from these gene ids
}
    
params_file = Path(json_output_path)
    
with open(params_file, "w") as f:
    json.dump(json_params_content, f, indent=4)

# run the printed command from any directory under the LOGIN node
# note that it also includes '-resume' argument to make sure that by default it won't be executed from the beginning
cmd = 'nextflow run '+\
subdirs['nanoflowz_dir']+'main.nf'+\
' -params-file '+str(params_file.resolve())+\
' -profile conda'+\
' -resume'
print(cmd)

## v2p0_ONT_FACSorganelles_Aleksei_v1

In [ ]:
WF_version = 'v2p0_ONT_FACSorganelles_Aleksei_v1'
dir_path = subdirs['wf_runs_dir']+WF_version+'/'

run_dir = subdirs['wf_runs_dir']+WF_version+'/'

# create a .toml file for polyA tail profiling
polyA_toml_output_path = os.path.join(dir_path,'polyA_config.toml')
(Path(polyA_toml_output_path).parent).mkdir(parents=True, exist_ok=True) # create subdir, just in case

config_data = {
    "tail": {
        "tail_interrupt_length": 0 # < maximum-strictness
    }
}
with open(polyA_toml_output_path, "wb") as f:
    tomli_w.dump(config_data, f)

# define pipeline configuration
json_output_path = dir_path+'run_params.json'
json_file_dir = Path(json_output_path).parent
json_file_dir.mkdir(parents=True, exist_ok=True)

# define custom parameters for nanoflowz run

json_params_content = {
    "tsv": str(Path(dir_path+'start_samples.tsv').resolve()),
    "rundir": str(Path(run_dir).resolve()),
    "dorado": str(file_paths['dorado_executor_v2_0']), # < newer version!
    "model": str(os.path.join(subdirs['dorado_models_dir'], 'dna_r10.4.1_e8.2_400bps_sup@v5.2.0')),
    "polyA": str(polyA_toml_output_path),
    "ref": str(os.path.join(run_dir,'reference_genome', 'genome_with_spikeINs.fasta')), # customly made genome which includes spike-ins
    "reference_gtf": str(os.path.join(run_dir,'reference_genome', 'compiled_genome_annotation.gtf')), # customly made genome which includes spike-ins
    "shift_ambiguous_cs": False, # <- here we don't enable the shifting of cleavage site positions!
    "outdir": os.path.join(run_dir,'results_alignment_with_SpikeIns'), # <- here !
    "check_gtf_compatibility": False, # no need, there are too many scaffolds in the genome that are not present in the used basic annotation
    "viz_gene_list": "ERCC-00130 ERCC-00096 ERCC-00002", # visualize raw signal from these gene ids
}
    
params_file = Path(json_output_path)
    
with open(params_file, "w") as f:
    json.dump(json_params_content, f, indent=4)

# run the printed command from any directory under the LOGIN node
# note that it also includes '-resume' argument to make sure that by default it won't be executed from the beginning
cmd = 'nextflow run '+\
subdirs['nanoflowz_dir']+'main.nf'+\
' -params-file '+str(params_file.resolve())+\
' -profile conda'+\
' -resume'
print(cmd)

# bulk RNA-seq from the same samples

## Prepare start samples for bulk RNA-seq from the same samples

In [ ]:
# we search for paths in the respective sequencing data folder
search_fastq_dir = os.path.join(subdirs['raw_sequencing_data_dir'],'20260218*')
os.system("""find """+search_fastq_dir+""" -name '*.fastq.gz' > """+subdirs['temp_dir']+"""FACS_organelles_samples.fastq_paths.tsv""")

fastq_file_paths = pd.read_csv(subdirs['temp_dir']+'FACS_organelles_samples.fastq_paths.tsv',delimiter="\t",
                                   index_col=None,header=None)
fastq_file_paths['merged_sample'] = fastq_file_paths.apply(lambda x:x[0].split('/')[-1].split('.')[0].split('_')[0],1)
fastq_file_paths['lane'] = fastq_file_paths.apply(lambda x:x[0].split('/')[-1].split('.')[0].split('_')[5],1)
fastq_file_paths['mate'] = fastq_file_paths.apply(lambda x:x[0].split('/')[-1].split('.')[0].split('_')[6],1)
fastq_file_paths = fastq_file_paths.sort_values(['merged_sample','lane','mate']).drop_duplicates(['merged_sample','lane','mate']).reset_index(drop=True)
fastq_file_paths['sample'] = fastq_file_paths.apply(lambda x:'_'.join([x['merged_sample'],x['lane']]),1)
fastq_file_paths = fastq_file_paths.rename(columns={0:"path"})

In [ ]:
fastq_file_paths['treat'] = fastq_file_paths.apply(lambda x:x['path'].split('-')[4],1).str.replace('si5','siCPSF5')
fastq_file_paths['fraction'] = fastq_file_paths.apply(lambda x:x['path'].split('-')[5].split('_')[0],1)
fastq_file_paths['condition'] = fastq_file_paths['treat']+';'+fastq_file_paths['fraction']
print(fastq_file_paths['treat'].unique())
print(fastq_file_paths['fraction'].unique())

In [ ]:
fastq_file_paths.iloc[16]['path']

In [ ]:
# Create symbolink link copies for all experimental fastq files

fastq_file_paths["start_file_path"] = (
    subdirs["fastq_dir"]
    + fastq_file_paths["sample"]
    + "."
    + fastq_file_paths["mate"]
    + ".fastq.gz"
)

# for index, row in fastq_file_paths.iterrows():
    # command = (
    #     "rm -f "
    #     + row["start_file_path"]
    #     + " && ln -f -s "
    #     + row["path"]
    #     + " "
    #     + row["start_file_path"]
    # )
    # out = subprocess.check_output(command, shell=True)

Here is the [protocol description for used Takara library prep](https://www.takarabio.com/products/next-generation-sequencing/rna-seq/total-rna-seq/smart-seq-total-rna-library-prep-with-zapr-depletion-(with-umis))

In [ ]:
WF_version = "RNAseq_FACSorganelles_Aleksei_v1"

dir_path = os.path.join(subdirs["wf_runs_dir"],WF_version)
command = "mkdir -p " + dir_path
out = subprocess.check_output(command, shell=True)

start_samples = fastq_file_paths.drop(["path"], axis=1).rename(
    columns={"start_file_path": "fq"}
)

start_samples = pd.merge(
    start_samples.loc[start_samples["mate"] == "R1"][
        ["sample", "merged_sample", "fq"]
    ].rename(columns={"fq": "fq1"}),
    start_samples.loc[start_samples["mate"] == "R2"][
        ["sample", "merged_sample", "fq"]
    ].rename(columns={"fq": "fq2"}),
    how="inner",
    on=["sample", "merged_sample"],
)
start_samples["fq1_3p"] = "AGATCGGAAGAG"  # forcely put Illumina universal adapter
start_samples["fq2_3p"] = "AGATCGGAAGAG"  # forcely put Illumina universal adapter

start_samples["fq2_UMI_len"] = "N" * 8  # first 8nts of read 2 are UMIs
start_samples["fq2_trim5"] = (
    3 + 3
)  # fixed number of nt to trim from 5'end of the R2 reads (template-switching part) AFTER UMI extraction
start_samples["fq1_trim5"] = 0  # nothing to trim

start_samples.to_csv(
k    os.path.join(dir_path, "start_samples.tsv"),
    sep=str("\t"),
    header=True,
    index=None,
    quoting=csv.QUOTE_NONE,
)

## Prepare .yaml config file and run snakemake WF

create conda environment with snakemake and install SLURM executor

run from the login node on HPC cluster:

```bash
conda create -c conda-forge -c bioconda -n snakemake snakemake
conda activate snakemake
pip install snakemake-executor-plugin-slurm
```

In [ ]:
organism = 'human'

gtf_chrs = pd.read_csv(file_paths[organism+'_basic_annotation_file'],delimiter="\t",
                                   index_col=None,header=None,usecols = [0],skiprows=5)
chromosome_list = list(gtf_chrs[0].unique())

In [ ]:
# load default rule_config, modify it and save
WF_version = "RNAseq_FACSorganelles_Aleksei_v1"

yaml = ruamel.yaml.YAML()
yaml.preserve_quotes = True
with open(subdirs["wf_dir"] + "config.yaml") as f_read:
    data = yaml.load(f_read)

data["samples_file"] = subdirs["wf_runs_dir"] + WF_version + "/start_samples.tsv"
data["output_dir"] = subdirs["wf_runs_dir"] + WF_version + "/output/"
data["local_log"] = subdirs["wf_runs_dir"] + WF_version + "/output/local_log/"
data["cluster_log"] = subdirs["wf_runs_dir"] + WF_version + "/output/cluster_log/"

data["organism"] = organism
data["genome_file"] = file_paths[organism+"_genome_file"]
data["gtf_file"] = file_paths[organism+'_basic_annotation_file']

data["chromosomes"] = " ".join(chromosome_list)

for dir_path in [data["output_dir"], data["local_log"], data["cluster_log"]]:
    command = "mkdir -p " + dir_path
    out = subprocess.check_output(command, shell=True)

with open(
    subdirs["wf_runs_dir"] + WF_version + "/modified_config_RNAseq.yaml", "w"
) as f_write:
    yaml.dump(data, f_write)

WF_step = "basic-TakaraPicoUMI-pe" # this workflow was optimized for the used kit

command = (
    """snakemake \
--snakefile """
    + subdirs["wf_dir"]
    + """Snakefile-"""
    + WF_step
    + """ \
--scheduler greedy \
--configfile """
    + subdirs["wf_runs_dir"]
    + WF_version
    + "/modified_config_RNAseq.yaml"
    + """ \
--printshellcmds \
--software-deployment-method conda apptainer \
--conda-frontend conda \
--apptainer-args "--bind """
    + subdirs["wf_dir"]
    + """,/scicore/home/zavolan/GROUP/"""
    + """" \
--executor slurm \
--profile """
    + subdirs["wf_dir"]
    + "profile"
    + """ \
--nolock \
-np"""
)

print(command)

# Analyze bulkRNA-seq in juxtaposition with ONT

## Define metadata labels and colors uniformly

In [ ]:
def get_metadata_with_labels_and_colors_bulkRNAseq(input_df):
    sel_files = input_df.copy()
    condition_dict = {
        "GFB-66210": "siCTRL Total",
        "GFB-66211": "siCPSF5 Total",
        "GFB-66212": "siCTRL Organelles",
        "GFB-66213": "siCPSF5 Organelles",
        "GFB-66214": "siCTRL TIS",
        "GFB-66215": "siCPSF5 TIS",
        "GFB-66216": "siCTRL ER",
        "GFB-66217": "siCPSF5 ER",
        "GFB-66218": "siCTRL TISnew",
        "GFB-66219": "siCPSF5 TISnew",
    }
    sel_files["condition"] = sel_files['merged_sample'].map(condition_dict)

    cat = "condition"
    cat_order = [condition_dict[elem] for elem in condition_dict]
    cat_palette = sns.color_palette("tab20")[:len(cat_order)]
    cat_color_dict = {}
    for k, cat_val in enumerate(cat_order):
        cat_color_dict[cat_val] = cat_palette[k]
    sel_files[cat + ";color"] = sel_files[cat].map(cat_color_dict)
    sel_files['condition'] = pd.Categorical(sel_files['condition'], categories=cat_order, ordered=True)
    
    sel_files = sel_files.sort_values(
        ["condition", "merged_sample", "sample"]
    ).reset_index(drop=True)
    return sel_files, cat_order, cat_palette

def get_metadata_with_labels_and_colors_ONT(input_df):
    sel_files = input_df.copy()
    condition_dict = {
        "barcode01": "siCTRL Total",
        "barcode03": "siCPSF5 Total",
        "barcode05": "siCTRL Organelles",
        "barcode06": "siCPSF5 Organelles",
        "barcode07": "siCTRL TIS",
        "barcode08": "siCPSF5 TIS",
        "barcode09": "siCTRL ER",
        "barcode10": "siCPSF5 ER",
        "barcode11": "siCTRL TISnew",
        "barcode12": "siCPSF5 TISnew",
        "barcode02": "siCTRL Total-LOW", # the same order as for bulk RNA-seq, but additional two categories
        "barcode04": "siCPSF5 Total-LOW",
    }
    sel_files["condition"] = sel_files['sample'].map(condition_dict)

    cat = "condition"
    cat_order = [condition_dict[elem] for elem in condition_dict]
    cat_palette = sns.color_palette("tab20")[:len(cat_order)]
    cat_color_dict = {}
    for k, cat_val in enumerate(cat_order):
        cat_color_dict[cat_val] = cat_palette[k]
    sel_files[cat + ";color"] = sel_files[cat].map(cat_color_dict)
    sel_files['condition'] = pd.Categorical(sel_files['condition'], categories=cat_order, ordered=True)
    
    sel_files = sel_files.sort_values(
        ["condition", "merged_sample", "sample"]
    ).reset_index(drop=True)
    return sel_files, cat_order, cat_palette

## Analysis of read mapping stats for bulk RNA-seq part

In [ ]:
WF_version = "RNAseq_FACSorganelles_Aleksei_v1"
organism = "human"
dir_path = subdirs["wf_runs_dir"] + WF_version + "/"

os.system(
    """find """
    + dir_path
    + "output/mapping_stats/"
    + """ -name '*.mapping_stats.txt' > """
    + subdirs["temp_dir"]
    + """mapping_stats.files.txt"""
)
os.system(
    """find """
    + dir_path
    + "output/mapping_stats/"
    + """ -name '*.success' > """
    + subdirs["temp_dir"]
    + """mapping_stats.success.files.txt"""
)

mapping_stats_files = pd.read_csv(
    subdirs["temp_dir"] + "mapping_stats.files.txt",
    delimiter="\t",
    index_col=None,
    header=None,
)
mapping_stats_success_files = pd.read_csv(
    subdirs["temp_dir"] + "mapping_stats.success.files.txt",
    delimiter="\t",
    index_col=None,
    header=None,
)

mapping_stats_files["sample"] = mapping_stats_files.apply(
    lambda x: x[0].split("/")[-1].replace(".mapping_stats.txt", ""), 1
)
mapping_stats_success_files["sample"] = mapping_stats_success_files.apply(
    lambda x: x[0].split("/")[-1].replace(".success", ""), 1
)

mapping_stats_files = mapping_stats_files.loc[
    mapping_stats_files["sample"].isin(
        list(mapping_stats_success_files["sample"].unique())
    )
].reset_index(
    drop=True
)  # look only at files with success flag

start_samples = pd.read_csv(
    dir_path + "start_samples.tsv", delimiter="\t", index_col=None, header=0
)

cat_name = "condition"
metadata_df, cat_order, cat_palette = get_metadata_with_labels_and_colors_bulkRNAseq(start_samples.copy())
metadata_df = metadata_df[['merged_sample','condition','condition;color']].rename(columns={'merged_sample':'sample'}).drop_duplicates() # we now get rid of 'merged_sample' definition

sel_files = pd.merge(
    mapping_stats_files,
    metadata_df.copy(),
    how="right",
    on=["sample"],
)
samples_list = list(sel_files["sample"])  # list of samples

In [ ]:
from zavolab_pyutils.parsing_workflow_outputs import (
    parse_mapping_stats
)

sub_sel_files = sel_files.copy()

i = 0
res = []
for index, row in sub_sel_files.iterrows():
    res_df = parse_mapping_stats(row[0],verbose=False)
    res_df["sample"] = row["sample"]
    res.append(res_df)
    if i % 2 == 0 and i != 0:
        print(f"{i} out of {len(sub_sel_files)} done")
    i = i + 1
mapping_stats_df = pd.concat(res).reset_index(drop=True)
numb_cols = list(mapping_stats_df.columns)[:4]
mapping_stats_df[numb_cols] = mapping_stats_df[numb_cols].astype('float')

mapping_stats_df["number of MM reads; genomic mapping"] = (
    mapping_stats_df["total number of mapped reads; genomic mapping"]
    - mapping_stats_df["number of UM reads; genomic mapping"]
)
numb_cols.append("number of MM reads; genomic mapping")

mapping_stats_df["number of MM reads; UMI dedup"] = (
    mapping_stats_df["total number of mapped reads; UMI dedup"]
    - mapping_stats_df["number of UM reads; UMI dedup"]
)
numb_cols.append("number of MM reads; UMI dedup")

for elem in numb_cols:
    mapping_stats_df[elem + "; mln"] = np.round(mapping_stats_df[elem] / 10**6, 2)

for read_cat in ['genomic mapping','UMI dedup']:
    mapping_stats_df['perc_of_UM; '+read_cat] = mapping_stats_df['number of UM reads; '+read_cat]/mapping_stats_df['total number of mapped reads; '+read_cat]*100
    mapping_stats_df['perc_of_MM; '+read_cat] = mapping_stats_df['number of MM reads; '+read_cat]/mapping_stats_df['total number of mapped reads; '+read_cat]*100

# add metadata for visualization
mapping_stats_df = pd.merge(mapping_stats_df,sel_files, how="left", on="sample")

In [ ]:
features = [
    "total number of mapped reads; genomic mapping; mln",
    "number of UM reads; genomic mapping; mln",
    "total number of mapped reads; UMI dedup; mln",
    "number of UM reads; UMI dedup; mln",
]
palette = ["black", "green", "grey", "royalblue"]

sns.set(font_scale=1)
sns.set_style("white")
fig, axes = plt.subplots(1, 1, sharey=False, sharex=True, figsize=(4, 4))

for i, feature in enumerate(features):

    x_feature, y_feature = feature, "condition"

    ax = sns.pointplot(
        data=mapping_stats_df, x=x_feature, y=y_feature, color=palette[i], label=feature
    )
    ax.legend(bbox_to_anchor=(1.05, 1.0), loc=2, borderaxespad=0.0, title="", ncols=1)
    # ax.set_yticklabels(mapping_stats_df['sample_label_within_experiment'])
    ax.tick_params(left=True, bottom=True)
ax.set(ylabel="", xlabel="# reads, mln")

out = subprocess.check_output(
    "mkdir -p " + subdirs["figures_dir"] + "mapping_stats/pilot_ER_TIS_Org/bulkRNAseq/", shell=True
)
fig.savefig(
    subdirs["figures_dir"] + "mapping_stats/pilot_ER_TIS_Org/bulkRNAseq/read_mapping.read_numbers.png",
    bbox_inches="tight",
    dpi=300,
)

In [ ]:
features = ["perc_of_UM; genomic mapping", "perc_of_UM; UMI dedup"]
palette = ["green", "royalblue"]

sns.set(font_scale=1)
sns.set_style("white")
fig, axes = plt.subplots(1, 1, sharey=False, sharex=True, figsize=(4, 4))

for i, feature in enumerate(features):

    x_feature, y_feature = feature, "condition"

    ax = sns.pointplot(
        data=mapping_stats_df, x=x_feature, y=y_feature, color=palette[i], label=feature
    )
    ax.legend(bbox_to_anchor=(1.05, 1.0), loc=2, borderaxespad=0.0, title="", ncols=1)
    ax.tick_params(left=True, bottom=True)
ax.set(ylabel="", xlabel="% of mapped reads")
out = subprocess.check_output(
    "mkdir -p " + subdirs["figures_dir"] + "mapping_stats/pilot_ER_TIS_Org/bulkRNAseq/", shell=True
)
fig.savefig(
    subdirs["figures_dir"] + "mapping_stats/pilot_ER_TIS_Org/bulkRNAseq/read_mapping.perc_of_UM.png",
    bbox_inches="tight",
    dpi=300,
)

## Obtaining and Analyzing gene expression counts - bulk RNAseq

In [ ]:
WF_version = "RNAseq_FACSorganelles_Aleksei_v1"
organism = "human"
dir_path = subdirs["wf_runs_dir"] + WF_version + "/"

os.system("""find """+subdirs['wf_runs_dir']+WF_version+'/output/gene_expression_quantification/'+\
          'FeatureCounts_genomic_DefaultGTF/'+""" -name '*.txt' > """+
          subdirs['temp_dir']+"""standard_FeatureCounts.files.txt""")

In [ ]:
entity_col = "sample" # should be a unique index for the metadata table

exonic_segments_FeatureCounts_files = pd.read_csv(subdirs['temp_dir']+'standard_FeatureCounts.files.txt',delimiter="\t",
                                   index_col=None,header=None)
exonic_segments_FeatureCounts_files[entity_col] = exonic_segments_FeatureCounts_files.apply(lambda x:x[0].split('/')[-1].replace('.txt',''),1)

exonic_segments_FeatureCounts_files = exonic_segments_FeatureCounts_files.rename(columns={0:'path'})

start_samples = pd.read_csv(
    dir_path + "start_samples.tsv", delimiter="\t", index_col=None, header=0
)

cat_name = "condition"
metadata_df, cat_order, cat_palette = get_metadata_with_labels_and_colors_bulkRNAseq(start_samples.copy())
metadata_df = metadata_df[['merged_sample','condition','condition;color']].rename(columns={'merged_sample':'sample'}).drop_duplicates() # we now get rid of 'merged_sample' definition

metadata_df = pd.merge(metadata_df,exonic_segments_FeatureCounts_files,how='inner',on=entity_col)

In [ ]:
i=0
for index,row in metadata_df.iterrows():
    tmp = pd.read_csv(row['path'],delimiter="\t",index_col=None,header=0,skiprows=1)
    cols = list(tmp.columns)
    tmp = tmp.rename(columns={cols[-1]:row[entity_col]})
    if i==0:
        res = tmp.copy()
    else:
        res = pd.merge(res,tmp,how='outer',on=cols[:-1])
    if i%5==0:
        print(str(i)+' done')
    i=i+1

In [ ]:
sel_metadata = metadata_df.copy()
sel_metadata = sel_metadata.rename(columns={entity_col:'sample'}) # rename to sample for consistency with downstream analysis
samples_list = list(sel_metadata['sample'])

index_cols = ['Geneid', 'Chr', 'Start', 'End', 'Strand', 'Length']
cur_res = res[index_cols+samples_list].reset_index(drop=True)
cur_res[samples_list] = cur_res[samples_list].fillna(0).astype('int')
cur_res = cur_res.rename(columns={'Geneid':'gene_id'}) # for easier correspondence with annotation file
index_cols = ['gene_id', 'Chr', 'Start', 'End', 'Strand', 'Length']

### Sanity style analysis

In [ ]:
# run additionally gtf parsing in case Deseq part was not used
from zavolab_pyutils.annotation import (
    parse_gtf_attributes_into_pd_dataframes,
)

gtf_df, genes_df, exons_df = parse_gtf_attributes_into_pd_dataframes(file_paths['human_basic_annotation_file'],
                                                                     input_skiprows=5,
                                                                     gene_type_field = "gene_type",
                                                                     extract_exon_number = True, 
                                                                     extract_gene_name_in_exons = True,
                                                                     verbose=False,
                                                                    )

In [ ]:
from zavolab_pyutils.read_count_data_analysis import (
    apply_deseq2_normalization, get_MultiDimR2,
    prepare_isoform_sanity_matrix, 
    apply_sanity_normalization_full_bayesian, 
    test_differential_relative_usage,
    test_differential_expression
)

In [ ]:
# we'll use all samples
sel_metadata = metadata_df.copy()

# adhere to sort order defined for cat_name
sel_metadata[cat_name] = pd.Categorical(sel_metadata[cat_name],categories=cat_order,ordered=True)
sel_metadata = sel_metadata.sort_values([cat_name,'sample']).reset_index(drop=True)

samples_list = list(sel_metadata['sample'])

In [ ]:
# we perform Sanity normalization and variance estimation
raw_counts_df = cur_res.loc[cur_res[samples_list].max(axis=1)>10].copy() # discard too lowly expressed genes
raw_counts_df.index = raw_counts_df['gene_id'].values

# Sanity-normalized counts are already log2-transformed
sanity_norm_counts_df, sanity_means_df, sanity_relative_errors_df, sanity_absolute_errors_df, sanity_vg_df, median_lib_size, variances_df = apply_sanity_normalization_full_bayesian(
    counts_df=raw_counts_df, 
    metadata_df=sel_metadata.copy(), 
    sample_col='sample', 
    cond_col='condition',
    vmin=0.0000001, 
    vmax=100,
    n_cores=12,
    empirical_bayes=False, # < Important!
    loess_variance_threshold_q=0.15,
)

In [ ]:
# Diagnostic plots for Sanity outputs

from zavolab_pyutils.visualization import (
    plot_variance_vs_expression,
    plot_mean_vs_cv,
)

# discard the effect from library size, as was done in original Sanity paper
ToCompare_sanity_norm_counts_df = sanity_norm_counts_df-np.log2(median_lib_size)

sanity_vg_df['inferred_v_g'] = sanity_vg_df['MAP_v_g'] # use MAP estimate for Vg to plot, in-line with original Sanity implementation
plot_variance_vs_expression(
    ToCompare_sanity_norm_counts_df, sanity_vg_df,
    savefig_path=subdirs['figures_dir']+'gene_expression_analysis/pilot_ER_TIS_Org/bulkRNAseq/pySanity/variance_vs_expr.png',
    true_vg=None, # This flag was used to useful
    ylim = (0,10),
)

natScale_ToCompare_sanity_norm_counts_df = 2**ToCompare_sanity_norm_counts_df

sel_metadata['all'] = 'all'
sanity_plot_data = plot_mean_vs_cv(
    natScale_ToCompare_sanity_norm_counts_df, sel_metadata.copy(),
    cond_col = 'all',
    savefig_path=subdirs['figures_dir']+'gene_expression_analysis/pilot_ER_TIS_Org/bulkRNAseq/pySanity/cv_vs_mean_plot.png')

In [ ]:
# Run PCA - now based on Sanity normalized counts

from zavolab_pyutils.visualization import (
    pca_plot
)

sanity_norm_counts_df['m'] = sanity_norm_counts_df[samples_list].mean(axis=1)
PCA_UMAP_subset = sanity_norm_counts_df.loc[sanity_norm_counts_df['m']>sanity_norm_counts_df['m'].quantile(0.1)].copy().reset_index(drop=True)
print(len(PCA_UMAP_subset))

savefig_path = (
    subdirs['figures_dir']+'gene_expression_analysis/pilot_ER_TIS_Org/bulkRNAseq/PCA_gene_expression.Sanity_norm_counts.png'
)

hue_column = cat_name
palette = cat_palette
hue_order = cat_order

pca_plot(
    PCA_UMAP_subset,
    samples_list,
    sel_metadata,
    "condition",
    savefig_path,
    sns_color_palette = palette,
    hue_order = hue_order,
    calculate_permanova_R2=False, # we don't have replicates, so makes no sense
    permanova_R2_ajusted = False,
    s_param=40,
    figsize=(4.2,4.2),
)

# we also save tsv-formatted data frames that were used to generate plots alongside with the figures
save_tsv_path = str(Path(savefig_path).with_suffix('.tsv'))
PCA_UMAP_subset.to_csv(save_tsv_path,sep=str("\t"),header=True,index=None,quoting=csv.QUOTE_NONE,)

In [ ]:
len(PCA_UMAP_subset)/len(cur_res.loc[cur_res[samples_list].max(axis=1)>0]) # fraction relative to expressed

## Obtaining and Analyzing gene expression counts - ONT

In [ ]:
WF_version = "v1p4_ONT_FACSorganelles_Aleksei_v2_full"
organism = "human"

command = 'find '+os.path.join(subdirs['wf_runs_dir'],WF_version,'results','read_tag_tables')+\
        " -name '*.tags.tsv.gz' > "+\
          os.path.join(subdirs['temp_dir'],"read_tag_tables.files."+WF_version+".txt")
out = subprocess.check_output(command, shell=True)

read_tag_tables_files = pd.read_csv(os.path.join(subdirs['temp_dir'],"read_tag_tables.files."+WF_version+".txt"),delimiter="\t",
                                   index_col=None,header=None)

entity_col = "sample" # should be a unique index for the metadata table
read_tag_tables_files[entity_col] = read_tag_tables_files.apply(lambda x:x[0].split('/')[-1].replace('.tags.tsv.gz',''),1)
read_tag_tables_files = read_tag_tables_files.rename(columns={0:'path'})

# load sample metadata
barcodes_metadata_df = pd.read_csv(subdirs['metadata_dir']+'barcodes_metadata.tsv',delimiter="\t",
                                   index_col=None,header=0)
barcodes_metadata_df['sample_id'] = barcodes_metadata_df.apply(lambda x:'barcode'+('0' if x['Barcode number']<10 else '')+str(x['Barcode number']),1)
barcodes_metadata_df['organism'] = barcodes_metadata_df.apply(lambda x:'human' if ('human' in x['species'].lower()) else 'drosophila',1)

barcodes_metadata_df['sample'] = barcodes_metadata_df['sample_id'] # for consistency with downstream processing

cat_name = "condition"
metadata_df, cat_order, cat_palette = get_metadata_with_labels_and_colors(barcodes_metadata_df.copy())
metadata_df = metadata_df[[entity_col,'condition','replicate','condition;color']].drop_duplicates().reset_index(drop=True)
metadata_df = pd.merge(metadata_df,read_tag_tables_files,how='inner',on=entity_col)

In [ ]:
# select chromosomes at which we want to look
chromosomes_pd = pd.read_csv(os.path.join(subdirs['wf_runs_dir'],WF_version,'reference_genome','compiled_genome_annotation.gtf'),delimiter="\t",
                                   index_col=None,header=None,usecols=[0])
chromosomes_pd = chromosomes_pd.drop_duplicates().reset_index(drop=True)

sel_chromosome_list = list(chromosomes_pd.loc[chromosomes_pd[0].str.startswith('chr')][0])

def natural_sort_key(s):
    parsed = re.split('([0-9]+)', str(s))
    return [(0, int(text)) if text.isdigit() else (1, text.lower()) for text in parsed]

sel_chromosome_list = sorted(sel_chromosome_list, key=natural_sort_key)

In [ ]:
i=0
res_dict = {}

for index,row in metadata_df.iterrows():
    tmp = pd.read_csv(row['path'],delimiter="\t",index_col=None,header=0,usecols=[1,2,3,4])
    cols = list(tmp.columns)

    tmp = tmp.loc[tmp['chromosome'].isin(sel_chromosome_list)].reset_index(drop=True)
    tmp['chromosome'] = pd.Categorical(tmp['chromosome'], categories=sel_chromosome_list, ordered=True)

    # only look at rows where pt was estimated
    tmp = tmp.loc[tmp['pt']>0]
    
    # calculate median-per-gene and abundance
    tmp['w'] = 1/tmp['NH'] # weight
    tmp['w_pt'] = tmp['w']*tmp['pt'] # we will calculate weighted mean
    
    
    index_cols = ['chromosome','XT']
    gr = tmp.groupby(index_cols).agg({'w':np.sum,'w_pt':np.sum}).reset_index()
    gr['pt_av'] = gr['w_pt']/gr['w']

    # we'll make a sample matrix for average polyA tail lengths and for abundance
    features = ['pt_av','w']
    for feature in features:
        if i==0:
            res_dict[feature] = gr[index_cols+[feature]].rename(columns={feature:row[entity_col]}).copy()
        else:
            res_dict[feature] = pd.merge(res_dict[feature],gr[index_cols+[feature]].rename(columns={feature:row[entity_col]}),
                                         how='outer',on=index_cols)
    if (i+1)%2==0:
        print(str(i)+' done')
    i=i+1

In [ ]:
sel_metadata = metadata_df.copy()
samples_list = list(sel_metadata['sample'])

In [ ]:
res_dict['w'][samples_list] = np.round(res_dict['w'][samples_list].fillna(0)).astype(int) # make integer counts